## RAG Application using Typesense

In [1]:
import typesense # type: ignore

In [ ]:
import os

client=typesense.Client({
  'nodes': [{
    'host': os.getenv('TYPESENSE_HOST', 'njky03hf7um8sqetp-1.a1.typesense.net'),
    'port': 443,
    'protocol': 'https'
  }],
  'api_key': os.getenv('TYPESENSE_API_KEY', 'f99UiF1mnhOhHCWRzGk755JOT1bZRjIt'),
  'connection_timeout_seconds': 2
})

books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}
try:
    client.collections['books'].delete()
except:
    pass

client.collections.create(books_schema) # type: ignore

ConfigError: `api_key` is not defined.

In [ ]:
client

In [ ]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()

client.collections['books'].documents.import_(data) # type: ignore

In [ ]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters) # type: ignore

In [ ]:
search_parameters = {
  'q'         : 'harry potter',
  'query_by'  : 'title',
  'filter_by' : 'publication_year:<1998',
  'sort_by'   : 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters) # type: ignore

In [ ]:
search_parameters = {
  'q'         : 'experyment',
  'query_by'  : 'title',
  'facet_by'  : 'authors',
  'sort_by'   : 'average_rating:desc'
}

client.collections['books'].documents.search(search_parameters) # type: ignore

### New Collection 

In [ ]:
### Langchain + Typsense + Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings # type: ignore
from langchain_groq import ChatGroq # type: ignore

In [ ]:
import os
# Load the GROQ API key from environment instead of hardcoding it.
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

In [ ]:
docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": os.getenv('TYPESENSE_HOST', 'njky03hf7um8sqetp-1.a1.typesense.net'),
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key": os.getenv('TYPESENSE_API_KEY', 'f99UiF1mnhOhHCWRzGk755JOT1bZRjIt'),
        "typesense_collection_name": "lang-chain"
    },
)

In [ ]:
query = "What is artificial intelligence"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

In [ ]:
### Retriever
retriever = docsearch.as_retriever()
retriever

In [ ]:
query = "Artificial intelligence indepth explanation"
retriever.invoke(query)[0]